# Q3 Editing Images with a Pretrained Flow Matching Model

In Q2 you trained a small class-conditional flow model on 64 x 64 images. In this notebook you will use a large *pretrained* text-to-image flow model, and edit images with it without any training: classifier-free guidance, SDEdit and visual anagrams.

We use **PixNerd** ([Wang et al., 2025](https://arxiv.org/abs/2507.23268), MIT license), a 1.2B parameter flow matching model that generates 512 x 512 images **directly in pixel space**, without a VAE, conditioned on text prompts that are encoded with the Qwen3-1.7B language model. Working in pixel space matters for the visual anagrams at the end: rotating an image means the same thing for the model as for us. PixNerd was trained with exactly the flow matching objective of Q1, with the same convention: $t = 0$ is noise $\epsilon \sim \mathcal{N}(0, I_d)$, $t = 1$ is the image $z$, $x_t = t\,z + (1 - t)\,\epsilon$, and the network predicts the velocity $u_t^\theta(x_t \mid y) \approx z - \epsilon$, where the condition $y$ is now a text prompt. So everything you learned in Q1 and Q2 applies directly.

## Workflow

1. **On your computer (no GPU needed)**: implement the TODOs in `flow_editing.py`, and test them with the test cells of this notebook (run the imports cell first, and skip the cells that use the model) or with the commands in the README. All tests run on the CPU in a few seconds, without the model. Also choose your own prompts and image in the cells marked `TODO`.
2. **On Delta AI (GPU)**: upload `flow_editing.py`, this notebook and your own image to your MP folder, and run the whole notebook as a batch job from the login node:
   ```bash
   sbatch run_notebook.sbatch Q3_flow_editing.ipynb
   ```
   The weights are already downloaded for you. When the job is done, download the figures in `outputs/` for your report. `Q3_flow_editing_run.ipynb` is a copy of this notebook with all its outputs, and the job log (`ece494mp5-mp5_notebook-<job id>.out`) shows any errors.
3. **Elsewhere**, on an NVIDIA GPU with about 4 GB of free memory, you can run this notebook as usual. The first time, loading the model downloads about 8 GB (PixNerd and its text encoder) to your Hugging Face cache. Every figure is saved to `outputs/`.

The model is wrapped in `pixnerd/__init__.py` (provided):
- `model.encode(prompt)` returns the embedding of a text prompt (cached in `pixnerd_prompt_embeds.pt`, so each prompt is only encoded once), and `model.null_emb` is the embedding of the empty prompt $\varnothing$,
- `model(x, t, emb)` predicts the velocity $u_t^\theta(x \mid y)$ for images `x` of shape `(B, 3, 512, 512)` with values in $[-1, 1]$.

The functions in `flow_editing.py` take two kinds of callables: a **model** `model(x, t, emb)`, and a **velocity field** `field(x, t)`, i.e. any function that maps `x` and `t` to a velocity, for example `field = lambda x, t: cfg_velocity(model, x, t, emb, null_emb, scale)`. The provided `euler_sample(field, x, ts)` integrates any velocity field, so guidance, SDEdit and anagrams all reuse the same sampler and only change the velocity field.

In [ ]:
# Uncomment the following pip command to install requirements
# NOT needed on Delta AI: run_notebook.sbatch uses the course virtual env
# ! pip install -r requirements.txt

In [ ]:
# loading autoreload extension for jupyter
%load_ext autoreload
try:
    # IPython 9+: always reload whole modules (its new partial reload can keep running your old code)
    %autoreload 2 --full
except Exception:
    # older IPython (e.g. on Delta AI) has no --full and always reloads whole modules
    %autoreload 2

In [ ]:
import os
import unittest

import numpy as np
import torch
from PIL import Image

from flow_editing import cfg_velocity, sdedit, anagram_velocity, euler_sample, get_times, rotate_180
from q2_helpers import get_device, randn, show_images

## Setup (GPU)

The cell below loads the model, and the test image. Sampling uses `N_STEPS` Euler steps for a full run from $t = 0$ to $t = 1$, along times from the provided `get_times`, which spends more steps at high noise (small $t$), as in the authors' sampler.

In [ ]:
from pixnerd import PixNerd

DEVICE = get_device()
print(f'Using device: {DEVICE}')
SHAPE = (3, 512, 512)
N_STEPS = 30   # Euler steps for a full sampling run from t = 0 to t = 1
SCALE = 4.0    # default guidance scale (the authors' default)
model = PixNerd(DEVICE)
NULL = model.null_emb


def load_image(path, res=512):
    """Load an image, center-crop it to a square, resize it to res x res, and scale it to [-1, 1]: (1, 3, res, res)."""
    im = Image.open(path).convert('RGB')
    s = min(im.size)
    im = im.crop(((im.width - s) // 2, (im.height - s) // 2, (im.width + s) // 2, (im.height + s) // 2))
    im = np.asarray(im.resize((res, res), Image.LANCZOS))
    return torch.from_numpy(im).permute(2, 0, 1)[None].float().to(DEVICE) / 127.5 - 1


def generate(emb, n, seed=0, scale=SCALE, n_steps=N_STEPS):
    """Generate n images for the prompt embedding emb from pure noise, with classifier-free guidance."""
    field = lambda x, t: cfg_velocity(model, x, t, emb, NULL, scale)
    return euler_sample(field, randn(n, seed, DEVICE, SHAPE), get_times(n_steps)).cpu()


foellinger = load_image('img/Foellinger.jpg')
show_images(foellinger, size=4)

# Part A: Sampling with Guidance

## Classifier-Free Guidance

As in Q2, we sample with classifier-free guidance, now with a text prompt $y$ as the condition and the empty prompt as $\varnothing$:

$$
\tilde{u}_t^\theta(x \mid y) = u_t^\theta(x \mid \varnothing) + w \left( u_t^\theta(x \mid y) - u_t^\theta(x \mid \varnothing) \right).
$$

The provided `cfg_velocity` in `flow_editing.py` computes this guided velocity: it is the same formula as your Q2.6 `get_velocity`, with prompt embeddings instead of class labels.

The cell below generates images from pure noise for two prompts, from the same noise, with three guidance scales: $w = 0$ ignores the prompt (unconditional samples), $w = 1$ is the conditional velocity $u_t^\theta(x \mid y)$ without guidance, and $w = 4$ is guided (the default scale of PixNerd).

In [ ]:
PROMPTS = ['a high quality photo', 'a photo of a red panda in a snowy forest']
for prompt in PROMPTS:
    emb = model.encode(prompt)
    images = torch.cat([generate(emb, 4, seed=2, scale=w) for w in [0.0, 1.0, SCALE]])
    show_images(images, ncols=4, size=2.5, row_labels=['w = 0', 'w = 1', f'w = {SCALE:g}'],
                fname=f"q3_cfg_{prompt.replace(' ', '_')[:40]}.png")

# Part B: Editing

## Q3.1 SDEdit

[SDEdit](https://arxiv.org/abs/2108.01073) edits an image with a generative model, without any extra training: noise the image back to some time $t_0$, and then run the sampler from $t_0$ to $1$. The noise destroys the details but keeps the coarse structure, so the sampler produces a new, realistic image with a similar structure that follows the prompt. In our convention, a **smaller** $t_0$ means **more** noise, i.e., more freedom to change the image.

**Q3.1 [1 pts Autograded]** Implement `sdedit`, using the provided `add_noise` and `euler_sample`. Once you have implemented the function, run the following cell to test your implementation.

In [ ]:
suite = unittest.TestLoader().loadTestsFromName('test_functions.TestClass.test_sdedit')
unittest.TextTestRunner().run(suite)

The cell below edits the test image with the prompt "a high quality photo" (which projects the image back onto natural images), and with a different prompt, for several start times $t_0$. All edits use the same noise.

In [ ]:
T_STARTS = [0.02, 0.05, 0.1, 0.15, 0.2, 0.3]


def edit_grid(z, emb, t_starts=T_STARTS, scale=SCALE, seed=1):
    """SDEdit of the images z with the prompt embedding emb: one row per image, one column per start time."""
    eps = randn(len(z), seed, DEVICE, SHAPE)
    field = lambda x, t: cfg_velocity(model, x, t, emb, NULL, scale)
    outputs = [sdedit(field, z, eps, get_times(N_STEPS, t_start=t0)) for t0 in t_starts]
    return torch.stack([z.cpu()] + [o.cpu() for o in outputs], dim=1).flatten(0, 1)


EDIT_PROMPTS = ['a high quality photo', 'a watercolor painting of a castle']  # TODO: try a prompt of your choice
for prompt in EDIT_PROMPTS:
    show_images(edit_grid(foellinger, model.encode(prompt)), ncols=1 + len(T_STARTS), size=2.5,
                titles=['input'] + [f't_0 = {t0}' for t0 in T_STARTS], row_labels=[prompt[:30]],
                fname=f"q3_1_sdedit_{prompt.replace(' ', '_')[:40]}.png")

Now try it on an image of your own: a photo, or a simple drawing (e.g. a sketch of a house or a face, drawn with a few colors). It is center-cropped to a square. On Delta AI, upload it to your MP folder together with this notebook.

In [ ]:
MY_IMAGE = None                    # TODO: path to your image, e.g. 'img/my_drawing.png'
MY_PROMPT = 'a high quality photo'  # TODO: a prompt that describes what the edited image should show

if MY_IMAGE is not None:
    show_images(edit_grid(load_image(MY_IMAGE), model.encode(MY_PROMPT)), ncols=1 + len(T_STARTS), size=2.5,
                titles=['input'] + [f't_0 = {t0}' for t0 in T_STARTS], fname='q3_1_sdedit_mine.png')

**Q3.2 [2 pts Manual graded]** Upload the visualizations above and answer the following:
1. How do the edits change with $t_0$? What is kept from the input image (e.g., layout, colors), and what changes? Which $t_0$ gives the best trade-off between realism and faithfulness to the input?
2. Describe your own image, your prompt and the result. Does the best $t_0$ differ from the test image? Why could that be?

# Part C: Combining Velocities

Because `euler_sample` integrates any velocity field, we can create new effects by combining the velocities for two prompts at every step.

## Q3.3 Visual Anagrams

**Visual anagrams** ([Geng et al., 2024](https://arxiv.org/abs/2311.17919)) look like one thing when upright and like another thing when rotated by 180°. At every step, we average the guided velocity for prompt $y_1$ on the image, and the guided velocity for prompt $y_2$ on the *rotated* image, rotated back so that it lines up with the image:

$$
u = \frac{1}{2} \left( \tilde{u}_t^\theta(x \mid y_1) + R\big(\tilde{u}_t^\theta(R(x) \mid y_2)\big) \right),
$$

where $R$ rotates by 180° (rotating twice gives back the original, so $R$ is its own inverse).

**Q3.3 [2 pts Autograded]** Implement `anagram_velocity`, using the provided `cfg_velocity` and the provided `rotate_180`. Once you have implemented the function, run the following cell to test your implementation.

In [ ]:
suite = unittest.TestLoader().loadTestsFromName('test_functions.TestClass.test_anagram_velocity')
unittest.TextTestRunner().run(suite)

In [ ]:
def show_anagram(prompt_1, prompt_2, seed=30, prefix='q3_3'):
    """Generate 4 visual anagrams for (upright, rotated) prompts, and show them upright and rotated."""
    emb_1, emb_2 = model.encode(prompt_1), model.encode(prompt_2)
    field = lambda x, t: anagram_velocity(model, x, t, emb_1, emb_2, NULL, SCALE)
    images = euler_sample(field, randn(4, seed, DEVICE, SHAPE), get_times(N_STEPS)).cpu()
    show_images(torch.cat([images, rotate_180(images)]), ncols=4, size=2.5, row_labels=['upright', 'rotated'],
                fname=f"{prefix}_anagram_{prompt_1.replace(' ', '_')[:30]}.png")


show_anagram('an oil painting of an old man', 'an oil painting of people around a campfire')

## Q3.4 Your Own Visual Anagram

Now surprise us: choose your own pair of prompts for a visual anagram, i.e. an image that clearly shows one thing upright and something unexpected when rotated. Some tips:
- Give both prompts the same style (e.g. "an oil painting of ...", "a watercolor of ...", "a lithograph of ..."): the image can only have one style, so the two prompts have to agree on it.
- Choose subjects whose shapes can match when rotated, e.g. a face and a landscape, or an animal and a building.
- Try several pairs, and several seeds for a promising pair: not every pair works, and not every sample of a good pair works.

**Q3.4 [4 pts Manual graded]** Upload the visualizations for the provided pair and for your own pair, and mark the best image of each. Then answer the following:
1. Why did you choose your prompts? Can you recognize both of them in your best image, upright and rotated, without knowing the prompts?
2. Describe a pair that you tried and that did not work, and include its visualization. What makes a pair work or fail? *Hint: at every step, the two prompts have to agree on every pixel, one on the upright image and one on the rotated image.*

In [ ]:
MY_ANAGRAM_PAIRS = []  # TODO: your pairs of prompts (upright, rotated), e.g. [('a watercolor of ...', 'a watercolor of ...')]
MY_SEED = 30          # TODO: try other seeds for a promising pair

for prompt_1, prompt_2 in MY_ANAGRAM_PAIRS:
    show_anagram(prompt_1, prompt_2, seed=MY_SEED, prefix='q3_4')